# Aumento de datos

**Capítulo 3 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_computer-vision/image-augmentation.ipynb` · [Lección original](https://d2l.ai/chapter_computer-vision/image-augmentation.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Aumento de la imagen
<a id="sec_image_augmentation"></a>

En [Referencia sec_alexnet](https://d2l.ai/chapter_convolutional-modern/alexnet.html#sec-alexnet), mencionamos que los grandes conjuntos de datos son un requisito previo para el éxito de las redes neuronales profundas en varias aplicaciones. *El aumento de imágenes* genera ejemplos de entrenamiento similares pero distintos después de una serie de cambios aleatorios en las imágenes de entrenamiento, ampliando así el tamaño del conjunto de entrenamiento. Alternativamente, el aumento de imágenes puede estar motivado por el hecho de que los ajustes aleatorios de ejemplos de entrenamiento permiten que los modelos confíen menos en ciertos atributos, mejorando así su capacidad de generalización. Por ejemplo, podemos recortar una imagen de diferentes maneras para que el objeto de interés aparezca en diferentes posiciones, reduciendo así la dependencia de un modelo en la posición del objeto. También podemos ajustar factores como el brillo y el color para reducir la sensibilidad de un modelo al color. Es probable que el aumento de imágenes fuera indispensable para el éxito de AlexNet en ese momento.


In [ ]:
%matplotlib inline
import torch
import torchvision
from torch import nn
from laboratorio import d2l

## Métodos comunes de aumento de la imagen
En nuestra investigación de métodos comunes de aumento de imágenes, utilizaremos la siguiente imagen $400\times 500$ como ejemplo.


In [ ]:
d2l.set_figsize()
img = d2l.Image.open('../recursos/originales/cat1.jpg')
d2l.plt.imshow(img);

La mayoría de los métodos de aumento de imágenes tienen un cierto grado de aleatoriedad. Para facilitarnos la observación del efecto del aumento de imágenes, a continuación definimos una función auxiliar `apply`. Esta función ejecuta el método de aumento de imágenes `aug` varias veces en la imagen de entrada `img` y muestra todos los resultados.


In [ ]:
def apply(img, aug, num_rows=2, num_cols=4, scale=1.5):
    Y = [aug(img) for _ in range(num_rows * num_cols)]
    d2l.show_images(Y, num_rows, num_cols, scale=scale)

### Flipping y Cropping

** Volcar la imagen a izquierda y derecha** generalmente no cambia la categoría del objeto. Este es uno de los métodos más antiguos y más ampliamente utilizados para aumentar la imagen. A continuación, utilizamos el módulo `transforms` para crear la instancia `RandomHorizontalFlip`, que gira una imagen a izquierda y derecha con un 50% de probabilidad.


In [ ]:
apply(img, torchvision.transforms.RandomHorizontalFlip())

** Voltear hacia arriba y hacia abajo** no es tan común como voltear a izquierda y derecha. Pero al menos para esta imagen de ejemplo, voltear hacia arriba y hacia abajo no impide el reconocimiento. A continuación, creamos una instancia `RandomVerticalFlip` para voltear una imagen hacia arriba y hacia abajo con un 50% de probabilidad.


In [ ]:
apply(img, torchvision.transforms.RandomVerticalFlip())

En la imagen de ejemplo que usamos, el gato está en el medio de la imagen, pero esto puede no ser el caso en general. En [Referencia sec_pooling](https://d2l.ai/chapter_convolutional-neural-networks/pooling.html#sec-pooling), explicamos que la capa de pooling puede reducir la sensibilidad de una capa convolucional a la posición de destino. Además, también podemos recortar aleatoriamente la imagen para hacer que los objetos aparezcan en diferentes posiciones en la imagen a diferentes escalas, lo que también puede reducir la sensibilidad de un modelo a la posición de destino.

En el siguiente código, **producimos aleatoriamente** un área con un área de $10\% \sim 100\%$ del área original cada vez, y la relación de anchura a altura de esta área se selecciona aleatoriamente de $0.5 \sim 2$. Luego, el ancho y la altura de la región se escalan a 200 píxeles. Salvo que se especifique lo contrario, el número aleatorio entre $a$ y $b$ en esta sección se refiere a un valor continuo obtenido por muestreo aleatorio y uniforme del intervalo $[a, b]$.


In [ ]:
shape_aug = torchvision.transforms.RandomResizedCrop(
    (200, 200), scale=(0.1, 1), ratio=(0.5, 2))
apply(img, shape_aug)

### Cambiar colores
Otro método de aumento es cambiar los colores. Podemos cambiar cuatro aspectos del color de la imagen: brillo, contraste, saturación y tono. En el siguiente ejemplo, ** cambiamos aleatoriamente el brillo** de la imagen a un valor entre 50% ($1-0.5$) y 150% ($1+0.5$) de la imagen original.


In [ ]:
apply(img, torchvision.transforms.ColorJitter(
    brightness=0.5, contrast=0, saturation=0, hue=0))

Del mismo modo, podemos ** cambiar aleatoriamente el tono** de la imagen.


In [ ]:
apply(img, torchvision.transforms.ColorJitter(
    brightness=0, contrast=0, saturation=0, hue=0.5))

También podemos crear una instancia `RandomColorJitter` y establecer cómo **cambiar aleatoriamente las `brightness`, `contrast`, `saturation` y `hue` de la imagen al mismo tiempo**.


### Nota docente de Hespérides

Para comparar modelos, conserva la partición y empareja las semillas. Selecciona hiperparámetros con validación y reserva el test para el final. La versión D2L de Fashion-MNIST llama «val» al test oficial: en estos derivados se separa validación del entrenamiento oficial. El modo rápido demuestra mecanismos; no permite extraer una clasificación definitiva de técnicas.

Vínculo con los apuntes: sesión 3, «Aumento de datos».


In [ ]:
color_aug = torchvision.transforms.ColorJitter(
    brightness=0.5, contrast=0.5, saturation=0.5, hue=0.5)
apply(img, color_aug)

### Combinación de múltiples métodos de aumento de imagen
En la práctica, **combinaremos múltiples métodos de aumento de imágenes**. Por ejemplo, podemos combinar los diferentes métodos de aumento de imágenes definidos anteriormente y aplicarlos a cada imagen a través de una instancia `Compose`.


In [ ]:
augs = torchvision.transforms.Compose([
    torchvision.transforms.RandomHorizontalFlip(), color_aug, shape_aug])
apply(img, augs)

## Formación con aumento de imagen

Entrenemos un modelo con aumento de imagen. Aquí usamos el conjunto de datos CIFAR-10 en lugar del conjunto de datos Fashion-MNIST que usamos anteriormente. Esto se debe a que la posición y el tamaño de los objetos en el conjunto de datos Fashion-MNIST se han normalizado, mientras que el color y el tamaño de los objetos en el conjunto de datos CIFAR-10 tienen diferencias más significativas. Las primeras 32 imágenes de entrenamiento en el conjunto de datos CIFAR-10 se muestran a continuación.


In [ ]:
all_images = torchvision.datasets.CIFAR10(train=True, root="../data",
                                          download=True)
d2l.show_images([all_images[i][0] for i in range(32)], 4, 8, scale=0.8);

Con el fin de obtener resultados definitivos durante la predicción, por lo general sólo aplicamos el aumento de imágenes a ejemplos de entrenamiento, y no utilizamos el aumento de imágenes con operaciones aleatorias durante la predicción. **Aquí sólo utilizamos el método de volteo aleatorio más simple a la izquierda-derecha**. Además, utilizamos una instancia `ToTensor` para convertir un minibatch de imágenes en el formato requerido por el biblioteca de aprendizaje profundo, es decir, números de 32 bits de punto flotante entre 0 y 1 con la forma de (tamaño del lote, número de canales, altura, anchura).


In [ ]:
train_augs = torchvision.transforms.Compose([
     torchvision.transforms.RandomHorizontalFlip(),
     torchvision.transforms.ToTensor()])

test_augs = torchvision.transforms.Compose([
     torchvision.transforms.ToTensor()])

A continuación, **definimos una función auxiliar para facilitar la lectura de la imagen y la aplicación de la ampliación de la imagen**. El argumento `transform` proporcionado por el conjunto de datos de PyTorch aplica la ampliación para transformar las imágenes. Para una introducción detallada a `DataLoader`, consulte [Referencia sec_fashion_mnist](https://d2l.ai/chapter_linear-classification/image-classification-dataset.html#sec-fashion-mnist).


In [ ]:
def load_cifar10(is_train, augs, batch_size):
    dataset = torchvision.datasets.CIFAR10(root="../data", train=is_train,
                                           transform=augs, download=True)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size,
                    shuffle=is_train, num_workers=d2l.get_dataloader_workers())
    return dataloader

### Entrenamiento de múltiples GPU
Entrenamos el modelo ResNet-18 desde
[Referencia sec_resnet](https://d2l.ai/chapter_convolutional-modern/resnet.html#sec-resnet) sobre el
Conjunto de datos CIFAR-10. Recordemos la introducción al entrenamiento multi-GPU en [Referencia sec_multi_gpu_concise](https://d2l.ai/chapter_computational-performance/multiple-gpus-concise.html#sec-multi-gpu-concise). En lo siguiente ** definimos una función para entrenar y evaluar el modelo utilizando múltiples GPU**.


In [ ]:
#@save
def train_batch_ch13(net, X, y, loss, trainer, devices):
    """Tren para un minibatch con múltiples GPU (definido en el capítulo 13)."""
    if isinstance(X, list):
        # Requerido para el ajuste fino de BERT (que se cubrirá más adelante)
        X = [x.to(devices[0]) for x in X]
    else:
        X = X.to(devices[0])
    y = y.to(devices[0])
    net.train()
    trainer.zero_grad()
    pred = net(X)
    l = loss(pred, y)
    l.sum().backward()
    trainer.step()
    train_loss_sum = l.sum()
    train_acc_sum = d2l.accuracy(pred, y)
    return train_loss_sum, train_acc_sum

In [ ]:
#@save
def train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs,
               devices=d2l.try_all_gpus()):
    """Formar un modelo con múltiples GPU (definido en el capítulo 13)."""
    devices = devices or [torch.device("cpu")]
    train_iter, test_iter, num_epochs = d2l.limitar_carga(train_iter, test_iter, num_epochs)
    timer, num_batches = d2l.Timer(), len(train_iter)
    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs], ylim=[0, 1],
                            legend=['train loss', 'train acc', 'test acc'])
    net = nn.DataParallel(net, device_ids=devices).to(devices[0])
    for epoch in range(num_epochs):
        # Suma de la pérdida de entrenamiento, suma de la exactitud de entrenamiento, no de ejemplos,
        # Número de predicciones
        metric = d2l.Accumulator(4)
        for i, (features, labels) in enumerate(train_iter):
            timer.start()
            l, acc = train_batch_ch13(
                net, features, labels, loss, trainer, devices)
            metric.add(l, acc, labels.shape[0], labels.numel())
            timer.stop()
            if (i + 1) % (max(1, num_batches // 5)) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (metric[0] / metric[2], metric[1] / metric[3],
                              None))
        test_acc = d2l.evaluate_accuracy_gpu(net, test_iter)
        animator.add(epoch + 1, (None, None, test_acc))
    print(f'loss {metric[0] / metric[2]:.3f}, train acc '
          f'{metric[1] / metric[3]:.3f}, test acc {test_acc:.3f}')
    print(f'{metric[2] * num_epochs / timer.sum():.1f} examples/sec on '
          f'{str(devices)}')

Ahora podemos **definir la función `train_with_data_aug` para entrenar el modelo con el aumento de imagen**. Esta función obtiene todas las GPU disponibles, utiliza Adam como algoritmo de optimización, aplica el aumento de imagen al conjunto de datos de entrenamiento, y finalmente llama a la función `train_ch13` que acaba de definirse para entrenar y evaluar el modelo.


In [ ]:
batch_size, devices, net = 256, d2l.try_all_gpus(), d2l.resnet18(10, 3)
net.apply(d2l.init_cnn)

def train_with_data_aug(train_augs, test_augs, net, lr=0.001):
    train_iter = load_cifar10(True, train_augs, batch_size)
    test_iter = load_cifar10(False, test_augs, batch_size)
    loss = nn.CrossEntropyLoss(reduction="none")
    trainer = torch.optim.Adam(net.parameters(), lr=lr)
    net(next(iter(train_iter))[0])
    train_ch13(net, train_iter, test_iter, loss, trainer, 10, devices)

Entrenemos el modelo usando el aumento de imagen basado en giros aleatorios izquierda-derecha.


In [ ]:
train_with_data_aug(train_augs, test_augs, net)

## Resumen
* El aumento de imágenes genera imágenes aleatorias basadas en datos de entrenamiento existentes para mejorar la capacidad de generalización de los modelos.
* Con el fin de obtener resultados definitivos durante la predicción, por lo general sólo aplicamos el aumento de imagen a ejemplos de entrenamiento, y no utilizamos el aumento de imagen con operaciones aleatorias durante la predicción.
* Los bibliotecas de aprendizaje profundo proporcionan muchos métodos diferentes de aumento de imágenes, que se pueden aplicar simultáneamente.

## Ejercicios
1. Capacitar el modelo sin usar el aumento de imagen: `train_with_data_aug(test_augs, test_augs)`. Comparar la precisión de entrenamiento y pruebas al usar y no usar el aumento de imagen. ¿Puede este experimento comparativo apoyar el argumento de que el aumento de imagen puede mitigar el exceso de ajuste?
1. Combine múltiples métodos de aumento de imágenes diferentes en el entrenamiento de modelos en el conjunto de datos CIFAR-10. ¿Mejora la precisión de las pruebas?
1. Consulte la documentación en línea del biblioteca de aprendizaje profundo. ¿Qué otros métodos de aumento de imágenes también proporciona?


[Debate del original](https://discuss.d2l.ai/t/1404)
